In [8]:
import os
import json
import cv2
import shutil
import random
from glob import glob
from tqdm import tqdm
from multiprocessing import Pool, cpu_count, freeze_support

# --- НАЛАШТУВАННЯ ---
DIRS_TO_PROCESS = [
    {"img": "dataset/images", "lbl": "dataset/annotations"},           # Чисті (Python)
    {"img": "unity/images", "lbl": "unity/labels"} # Unity
]

OUTPUT_BASE = "final_dataset"
YOLO_DIR = os.path.join(OUTPUT_BASE, "yolo")
CLASS_DIR = os.path.join(OUTPUT_BASE, "classifier")

def convert_to_yolo_bbox(box, img_w, img_h):
    dw = 1. / img_w
    dh = 1. / img_h
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]

    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return x, y, w, h

def process_single_pair(args):
    img_path, json_path, split_name, yolo_root, class_root = args
    local_counts = {}

    try:
        img = cv2.imread(img_path)
        if img is None: return {}
        h, w, _ = img.shape

        with open(json_path, 'r') as f:
            data = json.load(f)

        base_name = os.path.splitext(os.path.basename(img_path))[0]

        # --- YOLO ---
        dst_img = os.path.join(yolo_root, "images", split_name, base_name + ".jpg")
        cv2.imwrite(dst_img, img)

        yolo_txt_path = os.path.join(yolo_root, "labels", split_name, base_name + ".txt")
        with open(yolo_txt_path, 'w') as f_yolo:
            for item in data:
                zone_box = item.get("zone")
                if not zone_box: continue

                yx, yy, yw, yh = convert_to_yolo_bbox(zone_box, w, h)
                f_yolo.write(f"0 {yx:.6f} {yy:.6f} {yw:.6f} {yh:.6f}\n")

                # --- CLASSIFIER (ВИПРАВЛЕНО) ---
                if split_name in ["train", "val"]:
                    chars = item.get("chars", [])
                    for char_obj in chars:
                        # Отримуємо і тип, і сам символ
                        cls_type = char_obj.get("class_name") or char_obj.get("class")
                        char_symbol = char_obj.get("char_val") or char_obj.get("char") # Сама літера "5", "A"
                        bbox = char_obj.get("bbox")

                        if not bbox: continue

                        # ВИЗНАЧЕННЯ НАЗВИ ПАПКИ (LABEL)
                        folder_name = "tag" # За замовчуванням сміття

                        if cls_type == "digit":
                            folder_name = char_symbol # Папка буде "0", "1", "9" тощо
                        elif cls_type == "minus":
                            folder_name = "minus"
                        elif cls_type == "point":
                            folder_name = "point"
                        elif cls_type == "tag":
                            folder_name = "tag"

                        # Ігноруємо порожні або дивні символи
                        if not folder_name: continue

                        x1, y1, x2, y2 = map(int, bbox)
                        x1, y1 = max(0, x1), max(0, y1)
                        x2, y2 = min(w, x2), min(h, y2)

                        if x2 <= x1 or y2 <= y1: continue

                        char_crop = img[y1:y2, x1:x2]
                        if char_crop.size == 0: continue

                        # Зберігаємо в правильну папку (напр. classifier/train/5)
                        save_dir = os.path.join(class_root, split_name, folder_name)
                        try:
                            os.makedirs(save_dir, exist_ok=True)
                        except: pass

                        crop_name = f"{base_name}_{x1}_{y1}.jpg"
                        cv2.imwrite(os.path.join(save_dir, crop_name), char_crop)

                        count_key = f"{split_name}/{folder_name}"
                        local_counts[count_key] = local_counts.get(count_key, 0) + 1

    except Exception as e:
        return {}

    return local_counts

if __name__ == '__main__':
    freeze_support()

    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(YOLO_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(YOLO_DIR, "labels", split), exist_ok=True)

    all_pairs = []
    print("🔍 Збираємо файли...")
    for d in DIRS_TO_PROCESS:
        images = glob(os.path.join(d["img"], "*.jpg")) + glob(os.path.join(d["img"], "*.png"))
        for img_path in images:
            fname = os.path.basename(img_path)
            name_no_ext = os.path.splitext(fname)[0]
            json_path = os.path.join(d["lbl"], name_no_ext + ".json")
            if os.path.exists(json_path):
                all_pairs.append((img_path, json_path))

    random.seed(42)
    random.shuffle(all_pairs)

    train_end = int(len(all_pairs) * 0.8)
    val_end = int(len(all_pairs) * 0.9)
    splits = {
        "train": all_pairs[:train_end],
        "val": all_pairs[train_end:val_end],
        "test": all_pairs[val_end:]
    }

    tasks = []
    for split_name, pairs in splits.items():
        for img, js in pairs:
            tasks.append((img, js, split_name, YOLO_DIR, CLASS_DIR))

    cpu_cores = cpu_count()
    print(f"🚀 Починаємо обробку на {cpu_cores} ядрах...")

    final_char_counts = {}
    with Pool(cpu_cores) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_pair, tasks), total=len(tasks)))

    for res in results:
        for k, v in res.items():
            final_char_counts[k] = final_char_counts.get(k, 0) + v

    yaml_content = f"path: {os.path.abspath(YOLO_DIR)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: text_zone"
    with open(os.path.join(OUTPUT_BASE, "dataset_yolo.yaml"), 'w') as f:
        f.write(yaml_content)

    print("\n✅ Підготовка завершена!")
    print("Статистика класів:")
    # Сортуємо для красивого виводу (0-9, мінус, точка)
    for k, v in sorted(final_char_counts.items()):
        print(f"  {k}: {v}")

🔍 Збираємо файли...
🚀 Починаємо обробку на 4 ядрах...


100%|███████████████████████████████████████████████████████████████████████████████| 6988/6988 [04:14<00:00, 27.45it/s]



✅ Підготовка завершена!
Статистика класів:
  train/0: 5955
  train/1: 7771
  train/2: 7407
  train/3: 6795
  train/4: 8401
  train/5: 7821
  train/6: 5643
  train/7: 6228
  train/8: 5793
  train/9: 6549
  train/minus: 1823
  train/point: 7128
  train/tag: 30000
  val/0: 756
  val/1: 972
  val/2: 881
  val/3: 857
  val/4: 1039
  val/5: 913
  val/6: 695
  val/7: 800
  val/8: 696
  val/9: 825
  val/minus: 229
  val/point: 886
  val/tag: 3776
